<a href="https://colab.research.google.com/github/HoudiniJezewskii/ai-medical-support-chatbot/blob/main/Capstone_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q \
    "huggingface_hub>=0.25,<2.0" \
    transformers \
    accelerate \
    bitsandbytes \
    sentence-transformers \
    beautifulsoup4 \
    requests

In [ ]:
from huggingface_hub import login

login()

In [ ]:
from google.colab import userdata
import subprocess
import os

token = userdata.get("GITHUB_TOKEN")

if not token:
    raise ValueError("GITHUB_TOKEN was not found in Colab Secrets.")

repo_url = "https://github.com/HoudiniJezewskii/ai-medical-support-chatbot.git"
authenticated_url = repo_url.replace(
    "https://",
    f"https://x-access-token:{token}@"
)

%cd /content

if not os.path.exists("ai-medical-support-chatbot"):
    subprocess.run(
        ["git", "clone", authenticated_url],
        check=True
    )
    print("Repository cloned successfully.")
else:
    print("Repository already exists.")

%cd /content/ai-medical-support-chatbot

!git remote set-url origin https://github.com/HoudiniJezewskii/ai-medical-support-chatbot.git

!git config user.name "Your GitHub username"
!git config user.email "Your GitHub email"

/content
Repository cloned successfully.
/content/ai-medical-support-chatbot


In [ ]:
#load medgemma4
import torch
from transformers import (
    AutoProcessor,
    AutoModelForImageTextToText,
    BitsAndBytesConfig
)

model_name = "google/medgemma-4b-it"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

processor = AutoProcessor.from_pretrained(model_name)

model = AutoModelForImageTextToText.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto",
    low_cpu_mem_usage=True
)

print("MedGemma loaded successfully.")

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

MedGemma loaded successfully.


In [ ]:
import re
import requests
from bs4 import BeautifulSoup


SOURCE_PAGES = [
    (
        "MedlinePlus Common Cold",
        "https://medlineplus.gov/ency/article/000678.htm"
    ),
    (
        "MedlinePlus Common Cold Management",
        "https://medlineplus.gov/ency/patientinstructions/000466.htm"
    ),
    (
        "MedlinePlus Allergies",
        "https://medlineplus.gov/ency/article/000812.htm"
    ),
    (
        "MedlinePlus Medical Emergencies",
        "https://medlineplus.gov/ency/article/001927.htm"
    ),
    (
        "FDA Over-the-Counter Medicine Safety",
        "https://www.fda.gov/drugs/understanding-over-counter-medicines/"
        "over-counter-medicines-whats-right-you"
    ),
    (
        "FDA Acetaminophen Safety",
        "https://www.fda.gov/consumers/consumer-updates/"
        "dont-overuse-acetaminophen"
    )
]


def clean_html_text(html):
    """Extract readable text from an HTML page."""
    soup = BeautifulSoup(html, "html.parser")

    # Remove elements that do not contain useful article content.
    for tag in soup(["script", "style", "nav", "footer", "header", "aside"]):
        tag.decompose()

    main_content = soup.find("main") or soup.find("article") or soup.body

    if main_content is None:
        return ""

    text = main_content.get_text(" ", strip=True)
    return re.sub(r"\s+", " ", text)


def fetch_page(title, url):
    """Download and clean one trusted medical source."""
    response = requests.get(
        url,
        timeout=30,
        headers={
            "User-Agent": "Mozilla/5.0",
            "Accept": "text/html,application/xhtml+xml"
        }
    )

    response.raise_for_status()

    text = clean_html_text(response.text)

    if len(text) < 100:
        raise ValueError("The page returned too little readable text.")

    return {
        "title": title,
        "url": url,
        "text": text
    }


documents = []
failed_sources = []

for title, url in SOURCE_PAGES:
    try:
        document = fetch_page(title, url)
        documents.append(document)
        print(f"Loaded: {title}")

    except Exception as error:
        failed_sources.append({
            "title": title,
            "url": url,
            "error": str(error)
        })

        print(f"Could not load {title}: {error}")


print(f"\nLoaded {len(documents)} sources.")
print(f"Failed sources: {len(failed_sources)}")

if failed_sources:
    print("\nFailed sources:")
    for source in failed_sources:
        print(f"- {source['title']}: {source['error']}")

Loaded: MedlinePlus Common Cold
Loaded: MedlinePlus Common Cold Management
Loaded: MedlinePlus Allergies
Loaded: MedlinePlus Medical Emergencies
Loaded: FDA Over-the-Counter Medicine Safety
Loaded: FDA Acetaminophen Safety

Loaded 6 sources.
Failed sources: 0


In [ ]:
def create_chunks(text, chunk_size=300, overlap=50):
    words = text.split()
    chunks = []

    start = 0

    while start < len(words):
        end = start + chunk_size
        chunk_text = " ".join(words[start:end])

        if chunk_text.strip():
            chunks.append(chunk_text)

        start += chunk_size - overlap

    return chunks


chunks = []

for document in documents:
    document_chunks = create_chunks(document["text"])

    for chunk_number, chunk_text in enumerate(document_chunks):
        chunks.append({
            "text": chunk_text,
            "title": document["title"],
            "url": document["url"],
            "chunk_number": chunk_number
        })

print(f"Created {len(chunks)} chunks.")

Created 29 chunks.


In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

chunk_texts = [chunk["text"] for chunk in chunks]

chunk_embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

print(f"Embedding shape: {chunk_embeddings.shape}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding shape: (29, 384)


In [ ]:
def retrieve_context(question, number_of_results=3):
    question_embedding = embedding_model.encode(
        question,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    similarity_scores = chunk_embeddings @ question_embedding

    best_indices = np.argsort(similarity_scores)[::-1]
    best_indices = best_indices[:number_of_results]

    results = []

    for index in best_indices:
        result = chunks[index].copy()
        result["score"] = float(similarity_scores[index])
        results.append(result)

    return results

In [ ]:
test_question = "What should I do if I accidentally take too much acetaminophen?"

results = retrieve_context(test_question)

for result in results:
    print(f"Source: {result['title']}")
    print(f"Similarity score: {result['score']:.3f}")
    print(result["text"][:500])
    print("-" * 80)

Source: FDA Acetaminophen Safety
Similarity score: 0.656
Image Español You have flu symptoms, so you’ve been getting some relief by taking a cough and flu medicine every few hours. Late in the day, you have a headache and think about grabbing a couple of acetaminophen tablets (Tylenol, paracetamol and panadol) to treat the pain. Stop right there. You might not realize that more than 600 medications – both prescription and nonprescription (or over-the-counter, OTC) – contain acetaminophen to help relieve pain and reduce fever. Acetaminophen, either alo
--------------------------------------------------------------------------------
Source: FDA Acetaminophen Safety
Similarity score: 0.563
as many as 12% of people get the flu. Moreover, 7 in 10 people use nonprescription medicines to treat cold, cough and flu symptoms. Acetaminophen is often an ingredient in many medications used to treat these symptoms. Acetaminophen is used in many commonly prescribed medications in combination with pai

In [ ]:
def format_retrieved_context(question, number_of_results=3):
    retrieved_results = retrieve_context(
        question,
        number_of_results=number_of_results
    )

    context_sections = []

    for result in retrieved_results:
        context_sections.append(
            f"Source: {result['title']}\n"
            f"URL: {result['url']}\n"
            f"Information: {result['text']}"
        )

    return "\n\n".join(context_sections)


def ask_medgemma(question, conversation_history):
    retrieved_context = format_retrieved_context(question)

    system_instruction = """
You are a concise medical information chatbot.

Rules:
- Answer in no more than 3 short sentences.
- Do not use bullet points or headings.
- Do not say "Okay" or repeat the question.
- Do not diagnose.
- Do not prescribe medication.
- Do not provide extra home remedies.
- If serious symptoms or an emergency are mentioned, recommend immediate medical care.
- Use only information supported by the retrieved sources.
- Do not invent medical information.
- State that the information is general education.
"""

    messages = [
        {
            "role": "system",
            "content": [
                {
                    "type": "text",
                    "text": system_instruction
                }
            ]
        }
    ]

    messages.extend(conversation_history[-4:])

    augmented_question = f"""
Use the retrieved medical information below to answer the user's question.

Retrieved medical information:
{retrieved_context}

User question:
{question}
"""

    messages.append(
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": augmented_question
                }
            ]
        }
    )

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=70,
        do_sample=False,
        repetition_penalty=1.15,
        no_repeat_ngram_size=3,
        pad_token_id=processor.tokenizer.eos_token_id
    )

    answer_tokens = outputs[0][inputs["input_ids"].shape[-1]:]

    answer = processor.decode(
        answer_tokens,
        skip_special_tokens=True
    )

    return answer.strip()


def start_chatbot():
    print("AI Medical Support Chatbot")
    print("Type 'quit' to exit.\n")

    conversation_history = []

    while True:
        question = input("You: ").strip()

        if question.lower() in ["quit", "exit", "bye"]:
            print("Chatbot: Goodbye!")
            break

        if not question:
            continue

        answer = ask_medgemma(
            question,
            conversation_history
        )

        print("\nChatbot:")
        print(answer)
        print()

        conversation_history.append(
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": question
                    }
                ]
            }
        )

        conversation_history.append(
            {
                "role": "assistant",
                "content": [
                    {
                        "type": "text",
                        "text": answer
                    }
                ]
            }
        )

In [ ]:
start_chatbot()

AI Medical Support Chatbot
Type 'quit' to exit.

